# 🧹 MozDev — Script Automation Part 1 · Limpeza de Dados

**Objectivo:** Detectar e corrigir os 3 tipos de erros injectados nos ficheiros Excel:

| # | Tipo | Descrição | Função SQL |
|---|------|-----------|------------|
| 1 | `NEGATIVO` | Valores numéricos negativos | `ABS()` |
| 2 | `ESPACOS` | Espaços extra em strings de pagamento | `TRIM()` |
| 3 | `DATA_ERRADA` | Datetime em vez de date pura | `DATE()` |

> 🔵 Azul = input fixo · 🖤 Preto = fórmula · 🟢 Verde = ligação entre folhas

## ⚙️ Setup — instalar dependências e ligar ao Drive

In [1]:
# Célula 1 — instalar dependências
!pip install jupysql pandas openpyxl --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.5/268.5 kB 9.3 MB/s eta 0:00:00


In [2]:
# Célula 2 — importar bibliotecas
import pandas as pd
import sqlite3
from google.colab import drive
import os

In [3]:
# Célula 3 — montar Google Drive e definir pasta de trabalho
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/MozDev_AI_Automation_Scripts/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Drive montado em: {OUTPUT_DIR}")

Mounted at /content/drive
✅ Drive montado em: /content/drive/MyDrive/MozDev_AI_Automation_Scripts/


In [4]:
# Célula 4 — navegar para a pasta de trabalho
%cd /content/drive/MyDrive/MozDev_AI_Automation_Scripts/
%ls

/content/drive/MyDrive/MozDev_AI_Automation_Scripts
mozchapa100.db                MozChapa100_Segmentos_ERROS.xlsx
MozChapa100_Rotas_ERROS.xlsx  MozChapa100_Vendas_ERROS.xlsx


## 🗄️ Criar base de dados SQLite a partir dos ficheiros Excel

> 💬 **Prompt para gerar esta célula:**
>
> *"Tenho 3 ficheiros Excel: MozChapa100_Segmentos_ERROS.xlsx, MozChapa100_Vendas_ERROS.xlsx, MozChapa100_Rotas_ERROS.xlsx. Cada ficheiro possui um título decorativo na linha 0 e cabeçalhos reais na linha 1. Escreve código Python usando pandas e sqlite3 para carregar os 3 ficheiros numa base de dados SQLite chamada mozchapa100.db, cada um como a sua própria tabela com os nomes segmentos, vendas, rotas."*

✅ DB criada com 3 tabelas: segmentos | vendas | rotas


In [6]:
# Célula 6 — activar SQL magic (permite escrever SQL directamente nas células)
%load_ext sql
%sql sqlite:///mozchapa100.db

Connecting to 'sqlite:///mozchapa100.db'

## 🔗 Criar tabela analítica — JOIN das 3 tabelas

Antes de limpar, precisamos de juntar `vendas`, `segmentos` e `rotas` numa só tabela de análise.
Esta tabela ainda **tem os erros** — vamos detectá-los a seguir.

> **Porquê `CREATE TABLE` e não `CREATE VIEW`?**  
> Em SQLite, `CREATE VIEW` é só uma query guardada — não armazena dados.  
> `CREATE TABLE AS SELECT` cria uma cópia física dos dados, mais rápida para consultar e limpar.

> 💬 **Prompt para gerar esta célula:**
>
> *"Tenho 3 tabelas no SQLite: vendas (colunas: Data, ID_Segmento, ID_Rota, Forma_Pagamento, Bilhetes_Vendidos, Preco_Unitario_MZN), segmentos (ID_Segmento, Tipo_Passageiro), rotas (ID_Rota, Data, Origem, Destino, Viagens_Realizadas, Passageiros_Total, Tempo_Medio_Min). Escreve um SQL CREATE TABLE AS SELECT que junta as 3 tabelas e agrupa por data, tipo de passageiro, rota (Origem + ' - ' + Destino) e forma de pagamento. Agrega: SOMA de bilhetes e receita (bilhetes × preço), MÉDIA de preço, SOMA de viagens e passageiros, MÉDIA de duração. Chama a tabela MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS."*

Running query in 'sqlite:///mozchapa100.db'

++
||
++
++

> 💬 **Prompt para gerar esta célula:**
>
> *"Escreve uma query SQL para mostrar as primeiras 10 linhas de MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS para que eu possa inspecionar visualmente os dados à procura de erros."*

Running query in 'sqlite:///mozchapa100.db'

dia,segmento,rota,Forma_Pagamento,Bilhetes_Vendidos,Preco_Unitario_MZN,receita,viagens,passageiros,duracao_min
2026-03-01 00:00:00,Estudante,Maputo Centro - Matola,M-Pesa,460.0,25.0,11500.0,1380.0,64308.0,40.0
2026-03-01 00:00:00,Trabalhador,Matola - Boane,e-Mola,1472.0,75.0,110400.0,1472.0,59616.0,74.0
2026-03-01 00:00:00,Turista,Machava - Marracuene,e-Mola,-460.0,75.0,-34500.0,828.0,19872.0,71.0
2026-03-01 00:00:00,Turista,Maputo Centro - KaMavota,Numerário,1840.0,100.0,184000.0,1104.0,29164.0,21.0
2026-03-01 00:00:00,Turista,Maputo Centro - Matola,M-Pesa,1564.0,75.0,117300.0,1380.0,64308.0,40.0
2026-03-02 00:00:00,Estudante,Machava - Marracuene,Numerário,1288.0,50.0,64400.0,460.0,4876.0,75.0
2026-03-02 00:00:00,Estudante,Maputo Centro - KaMavota,M-Pesa,736.0,50.0,36800.0,1288.0,30176.0,32.0
2026-03-02 00:00:00,Estudante,Matola - Boane,Numerário,92.0,50.0,4600.0,1288.0,23644.0,55.0
2026-03-02 00:00:00,Trabalhador,Maputo Centro - KaMavota,e-Mola,736.0,50.0,36800.0,1288.0,30176.0,32.0
2026-03-02 00:00:00,Turista,Machava - Marracuene,M-Pesa,1288.0,75.0,96600.0,460.0,4876.0,75.0


---
# 🔍 Detectar os Erros

**Regra de ouro:** Antes de corrigir, sempre **confirmar** quantos registos têm erro  
e **verificar** que a detecção é precisa (não apanha falsos positivos).

## ✅ Erro 1 — NEGATIVO: valores numéricos < 0

Os erros negativos foram injectados apenas em:
- `Bilhetes_Vendidos` (tabela vendas)
- `Preco_Unitario_MZN` (tabela vendas)

A coluna `receita` é calculada (`Bilhetes × Preço`), por isso fica negativa em cascata quando um dos dois é negativo.
`viagens`, `passageiros` e `duracao_min` **nunca tiveram negativos** — não precisam de ser corrigidas.

> 💬 **Prompt para gerar esta célula:**
>
> *"Na tabela MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS, escreve uma query SQL para encontrar todas as linhas onde Bilhetes_Vendidos ou Preco_Unitario_MZN são negativos."*

Running query in 'sqlite:///mozchapa100.db'

dia,segmento,rota,Forma_Pagamento,Bilhetes_Vendidos,Preco_Unitario_MZN,receita,viagens,passageiros,duracao_min
2026-03-01 00:00:00,Turista,Machava - Marracuene,e-Mola,-460.0,75.0,-34500.0,828.0,19872.0,71.0
2026-03-05 00:00:00,Turista,Maputo Centro - KaMavota,e-Mola,1380.0,-35.0,-48300.0,1104.0,32476.0,21.0
2026-03-11 00:00:00,Trabalhador,Machava - Marracuene,Numerário,-644.0,50.0,-32200.0,920.0,13432.0,82.0
2026-03-17 00:00:00,Turista,Maputo Centro - Matola,e-Mola,736.0,-75.0,-55200.0,1196.0,7820.0,57.0
2026-03-24 00:00:00,Turista,Maputo Centro - Machava,e-Mola,-1196.0,100.0,-119600.0,920.0,3772.0,32.0
2026-04-04 00:00:00,Trabalhador,Machava - Marracuene,M-Pesa,552.0,-100.0,-55200.0,736.0,12420.0,77.0


> **Dica para estudantes:** Deves ver exactamente **6 linhas** — as mesmas injectadas no Part 0.  
> Se vires mais ou menos, algo correu mal no JOIN acima.

## ✅ Erro 2 — ESPAÇOS: espaços extra em `Forma_Pagamento`

Valores como `' M-Pesa '` ou `' e-Mola '` parecem iguais ao olho,  
mas numa query `WHERE Forma_Pagamento = 'M-Pesa'` **não são encontrados**.

**Como detectar:** comparar o valor original com a versão limpa pelo `TRIM()`

```sql
-- ❌ ERRADO: LIKE '%  %' procura dois espaços consecutivos no meio da string
--           nunca vai encontrar ' M-Pesa ' porque o espaço é só no início/fim
WHERE Forma_Pagamento LIKE '%  %'

-- ✅ CORRECTO: compara o valor com a versão sem espaços nas pontas
WHERE Forma_Pagamento <> TRIM(Forma_Pagamento)
```

> 💬 **Prompt para gerar esta célula:**
>
> *"Na tabela MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS, escreve uma query SQL para detectar linhas onde a coluna Forma_Pagamento tem espaços no início ou no fim. Usa TRIM() para comparar — não uses LIKE."*

Running query in 'sqlite:///mozchapa100.db'

dia,segmento,rota,Forma_Pagamento,Bilhetes_Vendidos,Preco_Unitario_MZN,receita,viagens,passageiros,duracao_min
2026-03-08 00:00:00,Estudante,Maputo Centro - Machava,e-Mola,368.0,35.0,12880.0,1472.0,67252.0,35.0
2026-03-21 00:00:00,Turista,Machava - Marracuene,e-Mola,828.0,100.0,82800.0,828.0,20976.0,85.0
2026-03-31 00:00:00,Estudante,Matola - Boane,M-Pesa,184.0,35.0,6440.0,1288.0,51520.0,64.0


> **Dica para estudantes:** Deves ver **3 linhas** — nota que na coluna `Forma_Pagamento`  
> o valor parece normal mas tem espaços invisíveis nas pontas: `' e-Mola '`

## ✅ Erro 3 — DATA_ERRADA: datetime em vez de date pura

As datas com erro foram guardadas como `'2026-03-01 00:00:00'` em vez de `'2026-03-01'`.
Em SQLite, as datas são texto — por isso a detecção é feita por comprimento ou padrão:

```sql
-- ❌ ERRADO: CAST(dia AS DATE) em SQLite não faz o que se espera
--           SQLite não tem tipo DATE nativo — o CAST devolve o mesmo texto
--           por isso a comparação CAST(dia AS DATE) <> dia é sempre falsa
WHERE CAST(dia AS DATE) <> dia

-- ✅ CORRECTO: uma date pura tem exactamente 10 caracteres (YYYY-MM-DD)
--             se tem mais de 10, tem hora incluída
WHERE LENGTH(dia) > 10
```

> 💬 **Prompt para gerar esta célula:**
>
> *"Na tabela MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS, a coluna 'dia' deve conter apenas datas no formato AAAA-MM-DD (10 caracteres). Escreve uma query SQL para detectar linhas onde 'dia' tem mais de 10 caracteres, o que significa que contém uma componente de hora. Mostra o valor de dia, o seu comprimento, segmento e rota."*

Running query in 'sqlite:///mozchapa100.db'

dia,comprimento,segmento,rota
2026-03-01 00:00:00,19,Estudante,Maputo Centro - Matola
2026-03-01 00:00:00,19,Trabalhador,Matola - Boane
2026-03-01 00:00:00,19,Turista,Machava - Marracuene
2026-03-01 00:00:00,19,Turista,Maputo Centro - KaMavota
2026-03-01 00:00:00,19,Turista,Maputo Centro - Matola
2026-03-02 00:00:00,19,Estudante,Machava - Marracuene
2026-03-02 00:00:00,19,Estudante,Maputo Centro - KaMavota
2026-03-02 00:00:00,19,Estudante,Matola - Boane
2026-03-02 00:00:00,19,Trabalhador,Maputo Centro - KaMavota
2026-03-02 00:00:00,19,Turista,Machava - Marracuene


> **Dica para estudantes:** Deves ver linhas com `comprimento = 19` (`'2026-03-01 00:00:00'`).  
> Repara que **todas as linhas têm este problema** — porque quando se faz o JOIN,  
> o pandas converte todas as datas para datetime ao ler o Excel.  
> A limpeza com `DATE()` resolve isso de uma vez para todas as linhas.

---
# 🧹 Criar Tabela Limpa — `MOZ_DEV_MOZCHAPPA100_CLEAN`

Agora que confirmámos todos os erros, criamos uma tabela limpa aplicando as correcções:  
- `DATE(dia)` → remove a hora, fica só `YYYY-MM-DD`
- `TRIM(Forma_Pagamento)` → remove espaços nas pontas
- `ABS(Bilhetes_Vendidos)` e `ABS(Preco_Unitario_MZN)` → corrige os valores negativos
- `ABS(receita)` → corrige em cascata (era negativa por causa dos dois acima)

> **Nota importante:** `viagens`, `passageiros` e `duracao_min` **não usam `ABS()`**  
> porque nunca tiveram erros negativos. Aplicar `ABS()` sem necessidade é mau hábito —  
> podes estar a esconder erros reais futuros.

> 💬 **Prompt para gerar esta célula:**
>
> *"Tenho a tabela MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS com 3 tipos de erros: valores negativos em Bilhetes_Vendidos e Preco_Unitario_MZN (e receita em cascata), espaços no início/fim em Forma_Pagamento, e datetime em vez de date na coluna 'dia'. Escreve um SQL CREATE TABLE AS SELECT chamado MOZ_DEV_MOZCHAPPA100_CLEAN que corrige os 3 erros: usa ABS() apenas nas colunas com negativos, TRIM() em Forma_Pagamento, e DATE() em dia. Não apliques ABS() a viagens, passageiros ou duracao_min — essas colunas não tinham erros."*

Running query in 'sqlite:///mozchapa100.db'

++
||
++
++

## ✔️ Verificar — confirmar que os erros foram corrigidos

Após criar a tabela limpa, **re-executar as 3 queries de detecção** na nova tabela.  
Todas devem devolver **zero linhas**.

> 💬 **Prompt para gerar esta célula:**
>
> *"Escreve uma query SQL para verificar que MOZ_DEV_MOZCHAPPA100_CLEAN não tem valores negativos em Bilhetes_Vendidos, Preco_Unitario_MZN ou receita. Retorna uma contagem única."*

Running query in 'sqlite:///mozchapa100.db'

negativos_restantes
0


> 💬 **Prompt para gerar esta célula:**
>
> *"Escreve uma query SQL para verificar que MOZ_DEV_MOZCHAPPA100_CLEAN não tem linhas onde Forma_Pagamento tem espaços no início ou no fim. Retorna uma contagem única."*

Running query in 'sqlite:///mozchapa100.db'

espacos_restantes
0


> 💬 **Prompt para gerar esta célula:**
>
> *"Escreve uma query SQL para verificar que MOZ_DEV_MOZCHAPPA100_CLEAN não tem linhas onde a coluna 'dia' contém uma componente de hora (comprimento > 10). Retorna uma contagem única."*

Running query in 'sqlite:///mozchapa100.db'

datas_com_hora
0


> ✅ **Resultado esperado:** os 3 COUNTs devem devolver `0`.  
> Se algum devolver > 0, a correcção correspondente não funcionou — revê a query de criação da tabela.

## 👁️ Visualizar tabela limpa final

> 💬 **Prompt para gerar esta célula:**
>
> *"Escreve uma query SQL para mostrar as primeiras 10 linhas de MOZ_DEV_MOZCHAPPA100_CLEAN para confirmar: datas sem hora, sem espaços na forma de pagamento, sem números negativos."*

Running query in 'sqlite:///mozchapa100.db'

dia,segmento,rota,Forma_Pagamento,Bilhetes_Vendidos,Preco_Unitario_MZN,receita,viagens,passageiros,duracao_min
2026-03-01,Estudante,Maputo Centro - Matola,M-Pesa,460.0,25.0,11500.0,1380.0,64308.0,40.0
2026-03-01,Trabalhador,Matola - Boane,e-Mola,1472.0,75.0,110400.0,1472.0,59616.0,74.0
2026-03-01,Turista,Machava - Marracuene,e-Mola,460.0,75.0,34500.0,828.0,19872.0,71.0
2026-03-01,Turista,Maputo Centro - KaMavota,Numerário,1840.0,100.0,184000.0,1104.0,29164.0,21.0
2026-03-01,Turista,Maputo Centro - Matola,M-Pesa,1564.0,75.0,117300.0,1380.0,64308.0,40.0
2026-03-02,Estudante,Machava - Marracuene,Numerário,1288.0,50.0,64400.0,460.0,4876.0,75.0
2026-03-02,Estudante,Maputo Centro - KaMavota,M-Pesa,736.0,50.0,36800.0,1288.0,30176.0,32.0
2026-03-02,Estudante,Matola - Boane,Numerário,92.0,50.0,4600.0,1288.0,23644.0,55.0
2026-03-02,Trabalhador,Maputo Centro - KaMavota,e-Mola,736.0,50.0,36800.0,1288.0,30176.0,32.0
2026-03-02,Turista,Machava - Marracuene,M-Pesa,1288.0,75.0,96600.0,460.0,4876.0,75.0


---
## 💾 Exportar para Excel — `MozChapa100_CLEAN.xlsx`

Por fim, exportamos a tabela limpa para Excel.  
Este ficheiro é o input do passo seguinte (análise com STELA no Excel / dashboard).

> 💬 **Prompt para gerar esta célula:**
>
> *"Escreve código Python usando jupysql e pandas para ler a tabela completa MOZ_DEV_MOZCHAPPA100_CLEAN do SQLite e exportá-la para um ficheiro Excel chamado MozChapa100_CLEAN.xlsx. Imprime o caminho de saída, o número de linhas e os nomes das colunas."*

Running query in 'sqlite:///mozchapa100.db'

✅ Ficheiro exportado: MozChapa100_CLEAN.xlsx
   Linhas: 513
   Colunas: ['dia', 'segmento', 'rota', 'Forma_Pagamento', 'Bilhetes_Vendidos', 'Preco_Unitario_MZN', 'receita', 'viagens', 'passageiros', 'duracao_min']


---
## 🏁 Resumo do que aprendemos neste notebook

| Passo | O que fizemos | Função SQL |
|-------|--------------|------------|
| 1 | Carregar Excel → SQLite | `pandas.to_sql()` |
| 2 | Juntar 3 tabelas | `JOIN ... ON` |
| 3 | Detectar negativos | `WHERE col < 0` |
| 4 | Detectar espaços | `WHERE col <> TRIM(col)` |
| 5 | Detectar datas com hora | `WHERE LENGTH(col) > 10` |
| 6 | Criar tabela limpa | `CREATE TABLE AS SELECT` com `ABS()`, `TRIM()`, `DATE()` |
| 7 | Verificar limpeza | `COUNT(*)` nas queries de detecção → devem dar 0 |
| 8 | Exportar para Excel | `df.to_excel()` |

> **Próximo passo → Part 3:** análise e criação do dashboard no Excel com STELA 📊